# Random Forest vs XGBoost Models #

In [ ]:
# Import packages for data manipulation
import pandas as pd
import numpy as np
import pickle

# Import packages for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Import packages for data modeling and evaluation
from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

# Import packages for Random Forest & XGBoost modeling
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import plot_tree
from xgboost import XGBClassifier, plot_importance


## Imports and data loading ##

Start by importing packages needed to build machine learning models to achieve the goal of this project.

In [ ]:
# Load dataset into dataframe and drop missing values
data = pd.read_csv("../data/tiktok_dataset.csv").dropna(axis=0).reset_index(drop=True)
data_subset = data.copy()
data_subset.head()


In [ ]:
# Check class balance

data_subset['claim_status'].value_counts()

## Feature engineering

In [ ]:
# Extract the length of each `video_transcription_text` and add this as a column to the dataframe
data_subset['text_length']= data_subset['video_transcription_text'].str.len()
data_subset['text_length'].head()

In [ ]:
# Calculate the average text_length for claims and opinions
data_subset.groupby('claim_status')['text_length'].mean()

In [ ]:
# Visualize the distribution of `text_length` for claims and opinions
sns.histplot(data=data_subset, x = 'text_length', hue = 'claim_status', bins =50, alpha = 0.5)
plt.show()

In [ ]:
# Create a copy of the X data

data_subset['claim_status'] = data_subset['claim_status'].map({'opinion': 0, 'claim': 1})
data_subset['claim_status'].head(5)
# Drop unnecessary columns
data_subset = data_subset.drop(axis=1, columns = ['#', 'video_id'])

# Encode target variable
data_subset = pd.get_dummies(data_subset, columns = ['verified_status', 'author_ban_status'])


# Split the data

In [ ]:
# Isolate target variable

y = data_subset['claim_status']

In [ ]:
# Isolate features
x = data_subset.copy()
x = x.drop(columns = ['claim_status','video_view_count', 'video_transcription_text', 'video_like_count', 'video_share_count', 'video_download_count', 'video_comment_count'],  axis=1)

# Display first few rows of features dataframe
x.head()

# Create train/validate/test sets

In [ ]:
# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.25, random_state = 0)


Split the training set into training and validation sets, 75/25, to result in a final ratio of 60/20/20 for train/validate/test sets.

In [ ]:
# Split the training data into training and validation sets
x_tra, x_val, y_tra, y_val = train_test_split(x_train, y_train, test_size= 0.25, random_state=0)

Confirm that the dimensions of the training, validation, and testing sets are in alignment.

In [ ]:
# Get shape of each training, validation, and testing set
train_x = x_train.shape
test_x = x_test.shape
train_y = y_train.shape
test_y = y_test.shape
val_x = x_val.shape
val_y = y_val.shape

print("The dimension of X training data", train_x)

print("The dimension of Y training data", train_y)

print("The dimension of X test data", test_x)

print("The dimension of Y test data", test_y)
print("The dimension of X validation data", val_x)
print("The dimension of Y validation data", val_y)

# Build a random forest model

In [ ]:
# Instantiate the random forest classifier
rf = RandomForestClassifier(random_state = 0)

# Create a dictionary of hyperparameters to tune
hyperparameters = {
    'max_depth':[2,5,8],
    'n_estimators': [50, 100, 200],
    'min_samples_leaf': [2 , 4],
    'min_samples_split':[2, 5 , 20],
    'max_features': ['sqrt'],
    'max_samples': [0.3, 0.4, 0.5]
    
}

# Define a list of scoring metrics to capture
scoring = ['f1', 'accuracy', 'precision', 'recall']

# Instantiate the GridSearchCV object
split = [0 if x in x_val.index else -1 for x in x_train.index]
custom_split = PredefinedSplit(split)
cv_rf = GridSearchCV(rf, hyperparameters, scoring = scoring, cv =custom_split, refit = 'recall')

In [ ]:
### Fit the model to the data 

cv_rf.fit(x_train, y_train)

In [ ]:
# Examine best recall score
cv_rf.best_score_

In [ ]:
# Examine best parameters

cv_rf.best_params_

We check the precision score to ensure the model isn't labeling everything as a claim. We do this by using the cv_results_ attribute of the fit GridSearchCV object, which returns a numpy array that can be converted to a pandas dataframe. Then, examine the mean_test_precision column of this dataframe at the index containing the results from the best model. This index can be accessed by using the best_index_ attribute of the fit GridSearchCV object.

In [ ]:
# Access the GridSearch results and convert it to a pandas df
result = pd.DataFrame(cv_rf.cv_results_)

# Examine the GridSearch results df at column `mean_test_precision` in the best index
           
best_estimaor_result = result.iloc[result['mean_test_recall'].idxmax(), :]

precision = best_estimaor_result.mean_test_precision

f1 = best_estimaor_result.mean_test_f1

accuracy = best_estimaor_result.mean_test_accuracy

recall = best_estimaor_result.mean_test_recall

print(precision, f1, accuracy, recall)

The model performs moderately well; it yielded a precision score of 0.6568 and a recall score of 0.8004. Having a recall score of 0.80 is very good overall and aligns with our primary business objective, as correctly identifying claims (minimizing false negatives) is much more important for user safety than misidentifying opinions. However, having a high recall score is most likely to come at the expense of a low precision score, which means human moderators will still need to spend more time reviewing some opinion videos.

# Build an XGBoost model¶

In [ ]:

# Instantiate the XGBoost classifier
xgb = XGBClassifier(objective='binary:logistic')

# Create a dictionary of hyperparameters to tune

xgbhyperparameter = {
    'max_depth': [4,5],
    'min_child_weight': [1,2],
    'learning_rate': [0.2, 0.3],
    'n_estimators': [20, 50]
}
# Define a list of scoring metrics to capture
scoring = ['f1', 'accuracy', 'precision', 'recall']

# Instantiate the GridSearchCV object
xgb_cv = GridSearchCV(xgb, xgbhyperparameter, scoring = scoring, refit = 'recall')

In [ ]:
# Fit the model to the data
xgb_cv.fit(x_train, y_train)


In [ ]:
# Examine best recall score
xgb_cv.best_score_

In [ ]:
# Examine best parameters
xgb_cv.best_params_

In [ ]:
# Access the GridSearch results and convert it to a pandas df
xgb_results = pd.DataFrame(xgb_cv.cv_results_)

best_estimator_xgbresult = xgb_results.iloc[xgb_results['mean_test_recall'].idxmax(), :]
# Examine the GridSearch results df at column `mean_test_precision` in the best index

precision = best_estimator_xgbresult.mean_test_precision
recall = best_estimator_xgbresult.mean_test_recall
f1 = best_estimator_xgbresult.mean_test_f1
accuracy = best_estimator_xgbresult.mean_test_accuracy

print(precision, recall, f1, accuracy)

# Evaluate model
Evaluate models against validation criteria.

Random forest
  
  Display the predictions on the validation set.

In [ ]:
# Use the random forest "best estimator" model to get predictions on the validation set
y_predict = cv_rf.best_estimator_.predict(x_val)

In [ ]:
# Display the predictions on the validation set
y_predict

In [ ]:
# Display the true labels of the validation set
y_val

Create a confusion matrix to visualize the results of the classification model.

In [ ]:
# Create a confusion matrix to visualize the results of the classification model

# Compute values for confusion matrix
cm = confusion_matrix(y_val, y_predict)
# Create display of confusion matrix using ConfusionMatrixDisplay()
disp = ConfusionMatrixDisplay(confusion_matrix= cm, display_labels = None)

# Plot confusion matrix
disp.plot()

# Display plot
plt.show()

In [ ]:
# Create a classification report
# Create classification report for random forest model

from sklearn.metrics import classification_report

print(classification_report(y_val, y_predict))

The classification report displays the evaluation metrics of both claim and opinion. Since misidentifying claims as opinion would have a more detrimental impact on the user's safety and TikTok's overall system operation, we would focus on the model's claim evaluation metrics. The classification report shows a claim prediction precision score of 0.67, a recall score of 0.76, and an F1-score of 0.72. These scores align perfectly with what the confusion matrix displays. The true positive has the highest value, which connotes the model correctly identifies 1399 claims. The true negative shows that the model correctly identifies 1066 as opinion. However, the model misclassifies 679 opinions as claims and 435 claims as opinions. This indicates that there could be more room for improvement, which could be through exploring and engineering more features and expanding hyperparameter ranges.

# XGBoost
evaluate the XGBoost model on the validat

In [ ]:
# Use the best estimator to predict on the validation data
y_predict = xgb_cv.best_estimator_.predict(x_val)
y_predict

In [ ]:
# Compute values for confusion matrix
xgb_cm = confusion_matrix(y_val, y_predict, labels = None)


# Create display of confusion matrix using ConfusionMatrixDisplay()
xgb_disp = ConfusionMatrixDisplay(confusion_matrix = xgb_cm, display_labels = None)

# Plot confusion matrix
xgb_disp.plot()

# Display plot
plt.show()


In [ ]:
# Create a classification report
print(classification_report(y_val, y_predict))

The XGBoost model results in a claim identification precision score of 0.70, recall score of 0.73, and F1-score of 0.72. While XGBoost yields a higher precision score, in this case we are concentrating on a higher recall score. Since the “RandomForestClassifier” model results in a higher recall score of 76%, the random forest model is the champion.

## Use champion model to predict on test data

In [ ]:
y_predict = cv_rf.best_estimator_.predict(x_test)

In [ ]:
# Compute values for confusion matrix
cm = confusion_matrix(y_test, y_predict)

# Create display of confusion matrix using ConfusionMatrixDisplay()
disp = ConfusionMatrixDisplay(confusion_matrix= cm, display_labels = None)

# Plot confusion matrix
disp.plot()

# Display plot
plt.show()

In [ ]:
imp = cv_rf.best_estimator_.feature_importances_

f_imp = pd.Series(imp, index= x.columns)

fig , ax  =plt.subplots()
f_imp.plot.bar()

The most predictive features are video transcription text length, followed by the author-ban-status-active. The results make sense as claims usually require more explanation, evidence, and context, which translates into longer transcriptions. The authors who report claims are most likely to be heavily scrutinized and placed under banned or under review status, making active status highly predictive, as it strongly signals the absence of a claim. The absence of the claim here is valuable as it allows the model to reveal patterns in the data[](http://

The purpose of building this model is to predict whether a video posted on TikTok is a claim or an opinion. After cleaning and analyzing the data, it was split into the target variable (y variable) and features(x variables) and into training, validation, and test data. The model, then, was instantiated, the hyperparameters were tuned, the evaluation metric scores were identified, and GridSearchCV was executed. Following that, the best hyperparameters were used for validation and picking the champion model (Random Forest) that has the highest recall score. The champion the random forest model) was tested using the test data, and it seems that it performs well overall.

Next step: There could be some features that might improve the model's performance. Since video transcription text is the most predictive feature, we could extract common words and sentiment scores among videos flagged as claims using n-grams.